[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-GITHUB-USERNAME/JAXCode/blob/master/templates/16_cross_entropy.ipynb)

# 🟢 Easy: Cross-Entropy Loss

*Training*
Implement **cross-entropy loss directly from logits** — the plain form, no extras.

$$\ell_i = -\log p_{i,t_i}, \qquad
p_{i,c} = \frac{e^{z_{i,c}}}{\sum_{k} e^{z_{i,k}}}$$

Return the **mean over the batch**.

### Rules
- Signature: `cross_entropy_loss(logits, targets)`
- `logits` is `(B, C)`, `targets` is `(B,)` of integer class ids; the output is a **scalar**
- Banned: `optax`, and `jax.nn.log_softmax` / `jax.nn.softmax` — those *are* the
  answer, not tools
- **`jax.scipy.special.logsumexp` is allowed**, and is the intended route
- Must stay finite at extreme logits, and work under `jit`

### Stay in log space
Substitute the softmax into $-\log p_{i,t_i}$ and it collapses:

$$\ell_i = -\log \frac{e^{z_{i,t_i}}}{\sum_k e^{z_{i,k}}}
= \log \sum_k e^{z_{i,k}} \;-\; z_{i,t_i}
= \operatorname{logsumexp}(z_i) - z_{i,t_i}$$

No `exp` survives. That is the whole trick, and it is what keeps the loss finite
when a logit is 1000 or a prediction is confidently wrong — `logsumexp` does the
max-shift for you. Compute the probability first and the intermediate `exp(z)`
overflows to `inf`, or $p_t$ underflows to `0.0` and `log(0) = -inf` poisons
every gradient. **b_14** is this same problem with `logsumexp` banned, where you
write that shift yourself and the failure modes get the full treatment.

### Gathering the target
$z_{i,t_i}$ is one entry per row. Careful: `logits[targets]` indexes **rows**,
not one column per row — it returns `(B, C)`, silently. What you want is
`logits[jnp.arange(B), targets]`, or `jnp.take_along_axis(logits,
targets[:, None], axis=-1)[:, 0]`.

A one-hot `einsum` gets the same answer, but it builds a `(B, C)` matrix of
zeros to multiply against — `B×C` work and memory to read `B` numbers, which at
vocabulary sizes is the difference between a loss that fits and one that does
not. (`einsum` cannot index for you: `einsum('...c,...->...', logits, targets)`
multiplies by the target *values* and sums over the classes.)

Once this passes: **`b_14`** removes `logsumexp`, then **`b_15`** adds label
smoothing and a padding mask.

In [ ]:
# Colab setup (no-op when running locally).
# jax-judge is not published on PyPI, so the judge is installed from the
# repo itself. Regenerate with JAXCODE_REPO=you/YourFork to point this at
# your own fork:  JAXCODE_REPO=you/JAXCode make notebooks
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q flax optax')
    get_ipython().run_line_magic(
        'pip', 'install -q git+https://github.com/YOUR-GITHUB-USERNAME/JAXCode.git')
except ImportError:
    pass

In [ ]:
import jax
import jax.numpy as jnp

print("JAX", jax.__version__, "|", jax.devices())

In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

import jax
import jax.numpy as jnp


def cross_entropy_loss(logits, targets):
    """Mean cross-entropy over the batch.

    Args:
        logits:  (B, C) unnormalised scores
        targets: (B,) integer class ids

    Returns:
        Scalar loss.
    """
    pass  # Replace this

In [ ]:
# 🔍 Scratch cell — poke at your implementation
import jax
import jax.numpy as jnp

# Uniform logits over 3 classes -> loss is exactly log(3).
print("uniform:", cross_entropy_loss(jnp.zeros((1, 3)), jnp.array([0])), "vs", jnp.log(3.0))

# Random data against the library log-softmax.
logits = jax.random.normal(jax.random.key(0), (4, 5)) * 3.0
targets = jnp.array([1, 2, 0, 4])
ref = -jnp.mean(jnp.take_along_axis(
    jax.nn.log_softmax(logits, axis=-1), targets[:, None], axis=-1))
print("mine:", float(cross_entropy_loss(logits, targets)), " ref:", float(ref))

# The indexing trap: these are NOT the same thing.
print("\nlogits[targets].shape      :", logits[targets].shape, " <- rows, wrong")
print("take_along_axis(...).shape :",
      jnp.take_along_axis(logits, targets[:, None], axis=-1)[:, 0].shape, " <- right")

# The stability trap: huge logits.
big = jnp.array([[1000.0, 0.0, 0.0]])
print("\nbig logits, correct class:", cross_entropy_loss(big, jnp.array([0])))
print("naive softmax-then-log would give:",
      -jnp.log(jnp.exp(big) / jnp.exp(big).sum(-1, keepdims=True))[0, 0])

# Confidently WRONG stays finite — the loss is ~1000, not inf.
print("big logits, wrong class:  ", float(cross_entropy_loss(big, jnp.array([1]))))

# The gradient is p - onehot, averaged over the batch.
g = jax.grad(cross_entropy_loss)(jnp.array([[2.0, 1.0, 0.0]]), jnp.array([0]))
print("\ngrad:", g, " sums to", float(jnp.sum(g)))

In [ ]:
# ✅ SUBMIT — run this cell to check your solution
from jax_judge import check, hint, solution, status

check("cross_entropy")

# hint("cross_entropy")      # stuck? nudge without the answer
# solution("cross_entropy")  # spoiler: the reference implementation
# status()                   # your dashboard across all problems